# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook demonstrates step-by-step loading, exploration, and processing of the FAIR² dataset using the `mlcroissant` library. All references to dataset entities follow their Croissant `@id` fields.

### Dataset Source
The dataset is defined via a Croissant schema at:
[https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json)

**About this dataset:**
- Structured clinical, hormone, imaging, and genetic variant data for three female patients diagnosed with non-classical 21-hydroxylase deficiency-associated infertility.
- Multiple record sets: (1) clinical features (Table 1), (2) hormone & imaging findings (Table 2), (3) genetic variants (Table 3).

All data manipulations, as per Croissant, reference entities by their `@id`.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset package using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset package
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Here we inspect the Croissant schema for record sets, their IDs, and get field details for further extraction.

In [ ]:
# List available record sets and their @ids
record_sets = list(dataset.record_sets())

print("Available record sets by @id:")
for rs in record_sets:
    print(f"- {rs['@id']} : {rs.get('name', '(no label)')}")

# List fields for each record set (by @id)
for rs in record_sets:
    print(f"\nFields for record set @id={rs['@id']}:")
    fields = rs.get('field', [])
    if isinstance(fields, dict): fields = [fields]
    for field in fields:
        print(f"  - {field['@id']} : {field.get('name', '(no label)')}")

## 3. Data Extraction
Load tabular data from each record set into Pandas DataFrames for analysis using Croissant `@id` references.

First, define the record set `@id`s discovered above.
Then use `mlcroissant` to load each as a DataFrame.

In [ ]:
# Example record set @ids found above:
# - 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table1' (Clinical features)
# - 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table2' (Hormone levels & imaging)
# - 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table3' (Genetic variants)

# Replace with actual @id from previous step if different
record_set_ids = [
    'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table1',
    'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table2',
    'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table3'
]

dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns for record set {record_set_id}:")
    print(df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Process, filter, and normalize selected columns using field `@id`s. This demonstrates Croissant-driven referencing.

For illustration:
- We'll filter Table 1 data by patient age
- Normalize the `age_at_diagnosis` column (by its field `@id`)
- Group by a categorical variable (`@id` for 'infertility')

In [ ]:
# Select record set and fields by @id
table1_id = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table1'

# Example field @ids (actual @ids may be:
#   'https://sen.science/doi/10.71728/senscience.cbsv-djdb/field/age_at_diagnosis'
#   'https://sen.science/doi/10.71728/senscience.cbsv-djdb/field/infertility'
# Adjust to match listing in Section 2)
numeric_field_id = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/field/age_at_diagnosis'
group_field_id = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/field/infertility'

df = dataframes[table1_id]

# Filtering
threshold = 25
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize age distribution and the relationship between age and infertility using `@id` columns. Referencing by `@id` ensures consistency with Croissant metadata.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot: Age histogram
plt.figure(figsize=(6,3))
sns.histplot(df[numeric_field_id], bins=5, kde=True)
plt.xlabel("Age at Diagnosis")
plt.title("Distribution of Age at Diagnosis")
plt.show()

# Plot: Age by Infertility
plt.figure(figsize=(6,3))
sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
plt.xlabel("Infertility status")
plt.ylabel("Age at Diagnosis")
plt.title("Age by Infertility (@id-based referencing)")
plt.show()

## 6. Conclusion
This notebook illustrated FAIR² dataset exploration and processing via the `mlcroissant` library, strictly referencing Croissant entities (`recordSet`, `field`, `column`) by their `@id`. Using this approach:
- Loaded structured clinical, hormone, imaging, and genetic variant data for rare disease infertility analysis
- Demonstrated filtering, normalization, grouping, and visualization referencing Croissant schema

The FAIR² package demonstrates highly interoperable, schema-driven, entity-referenced workflows for biomedical datasets.

**Note:** Small sample size and anonymization limit broader statistical analysis, but the workflow is reproducible for larger Croissant-based datasets.